# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
from skewerjax.skewers.skewers import SkewerBundle
from skewerjax.cosmology.cosmology import Cosmology

from skewerjax.fields.fields import ScalarField, VectorField, Fields
import skewerjax.fields.toy_model_samplers as toy_samplers
import skewerjax.utils.aux as aux

from skewerjax.optical_depth import line_profile
import skewerjax.optical_depth.optical_depth as od

from skewerjax.constants.physical_constants import phys
from skewerjax.optical_depth.voigt import Lorentzian

# Validation with $T=0,\, v_{LoS}=0, \, n=const, a(x) = 1.0, a_0 = 1.0$

### Define the fields to sample

In [ ]:
L_box = 1.0

T0 = jnp.finfo(jnp.dtype('float32')).tiny
v0 = 0.0

fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=T0, T0_max=T0, L_box=L_box)),
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=v0, v0_max=v0, L_box=L_box)),
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0)),
        "number_density": ScalarField(toy_samplers.number_density_sampler(n0_min=1e2, n0_max=1e8, L_box=L_box)),
    },
    "SiIII": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=T0, T0_max=T0, L_box=L_box)),
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=v0, v0_max=v0, L_box=L_box)),
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0)),
        "number_density": ScalarField(toy_samplers.number_density_sampler(n0_min=1e2, n0_max=1e8, L_box=L_box)),
    }
})

### Define wavelength range and lines to study

In [ ]:
wavelengths = jnp.linspace(970, 1230, 2**19)  # in Angstroms
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"])

### Define skewers

In [ ]:
N_sk = 7
N_ppsk = 32

_points        = aux.make_plane_points(L_box, 'x', L_box/2, N_sk, N_ppsk)
skewer_bundle = SkewerBundle(
    spatial_points=_points,
    a0=1.0,                                                      # scale factor at which ICs are defined, default: 1.0
    x0=0.0,                                                      # comoving distance at which ICs are defined, default: 0.0
    ICs_defined_at_index=-1,                                     # indices along the skewer where ICs are defined, (-1 means last point, 0 means first point), default: -1
    ICs_at_emission_or_absorption="absorption",                  # whether the ICs are defined at emission or absorption, default: 'absorption'
    los_directions=None,                                         # Let SkewerBundle compute los_directions automatically
    manual_scale_factors=jnp.ones_like(_points[None, ..., 0]),   # if provided, must have shape (N_cosmologies, N_skewers, N_points_per_skewer)
    cosmologies=None                                             # when the manual scale factors are not provided, cosmologies must be defined
)

### Compute optical depth

In [ ]:
optical_depth = od.compute_optical_depth(
    skewer_bundle,
    wavelengths,
    line_bundle,
    fields,
    compute_mode="vmap",
    num_gpus=None
)
F_over_F0 = jnp.exp(-optical_depth)

### Analytic benchmark

In [ ]:
_scale_factor = 1.0
_scale_factor_obs = 1.0

gamma_GHz = line_bundle.A21_GHz / (4.0 * jnp.pi)

lambdas_A_prime = line_profile.compute_lambda_prime(
    wavelengths,
    jnp.array([_scale_factor_obs]),
    scale_factor=jnp.array([_scale_factor]),
    LoS_velocity=jnp.zeros(line_bundle.n_lines)[:, None]
)[:, 0]

lambda0_A = jnp.asarray(line_bundle.lambda0_A)  # (N_lines,)
nu_GHz = ((1.0 / lambdas_A_prime) - (1.0 / lambda0_A[:, None])) * phys.c * 10.0

phi_analytic = Lorentzian(nu_GHz, gamma_GHz[:, None])

cte_nm2_per_ns = (phys.qe**2 / (4 * phys.epsilon_0 * phys.me * phys.c))*1e3          # UNITS: [nm² / ns]
sigma_nm2 = cte_nm2_per_ns * line_bundle.f_osc[:, None] * phi_analytic               # UNITS: [nm²]

quadri_vectors_analytic = jnp.array([
    jnp.ones(N_sk) * _scale_factor_obs,
    jnp.ones(N_sk) * L_box / 2,
    jnp.linspace(0, L_box, N_sk),
    jnp.ones(N_sk) * L_box / 2,    
]).T
indices_for_species = {
    sp: line_bundle.indices_for_species(sp) for sp in line_bundle.species_keys
}
nunmber_densities_analytic = fields.tile_field_to_lines(
    "number_density",
    quadri_vectors_analytic,
    indices_for_species=indices_for_species,
)

optical_depth_analytic = sigma_nm2[:, None, :] * _scale_factor**-2 * nunmber_densities_analytic[:, :,  None] * L_box

F_over_F0_analytic = jnp.exp(-optical_depth_analytic)

### Validation plot

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)           # (N_lambda,)
F_over_F0_np = np.asarray(F_over_F0)                           # (N_sp, N_sk, N_lambda)
F_over_F0_analytic = np.asarray(F_over_F0_analytic)            # (N_sp, N_sk, N_lambda)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(line_bundle.lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    for i_sk in range(N_sk):
        ax.plot(lambdas_np, F_over_F0_np[i_sp, i_sk, :], color=colors[i_sk], linestyle='-', linewidth=2.0)
        ax.plot(lambdas_np, F_over_F0_analytic[i_sp, i_sk, :], color=colors[i_sk], linestyle=':', linewidth=6.0)

# Axes formatting
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$F(\lambda) / F_0$")
ax.set_title("Line profiles for different transitions", fontsize=20)

# handles = []
# for i_sk in range(N_sk):
#     _sampled_n = fields.number_density(skewer_bundle.quadri_vectors[i_sk])
#     lbl = rf"$n = {_sampled_n[0, 0]:.2g}\,\mathrm{{cm^{{-3}}}}$"
#     handles.append(Line2D([0], [0], color=colors[i_sk], lw=4, linestyle="-", label=lbl))
# leg = ax.legend(handles=handles, loc="upper left", fontsize=16, ncols=1)
# ax.add_artist(leg)

fig.tight_layout()
plt.show()




# Zoom half-width around each line center [Angstrom]
delta_lambda_A = 0.01   # adjust to taste
N_lines = line_bundle.n_lines
fig, axes = plt.subplots(
    nrows=2, ncols=N_lines, 
    figsize=(14, 8), 
    sharex='col',
    sharey='row',
    gridspec_kw=dict(height_ratios=[2, 1])
)

if N_lines == 1:
    axes = axes.reshape(2, 1)

for i_sp, (ax_main, ax_res, line) in enumerate(zip(axes[0], axes[1], line_bundle.lines)):
    lambda0 = line.lambda0_A
    
    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]

    ax_main.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)

    for i_sk in range(N_sk):
        ax_main.plot(lam_win, F_over_F0_np[i_sp, i_sk, mask], color=colors[i_sk], linestyle='-', linewidth=2.0)
        ax_main.plot(lam_win, F_over_F0_analytic[i_sp, i_sk, mask], color=colors[i_sk], linestyle=':', linewidth=6.0, alpha=0.6)

        residual = F_over_F0_np[i_sp, i_sk, mask] / F_over_F0_analytic[i_sp, i_sk, mask] - 1
        ax_res.plot(lam_win, residual, color=colors[i_sk], linestyle='-', linewidth=2.0)

    ax_main.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax_main.set_ylabel(r"$F(\lambda) / F_0\, \left[\mathrm{adim}\right]$")
    ax_res.axhline(0.0, color="k", linewidth=1, alpha=0.5)
    ax_res.fill_between(lam_win, -1e-6, 1e-6, color="grey", alpha=0.2)
    ax_res.hlines(1e-6, xmin=lam_win[0], xmax=lam_win[-1], color="k", linewidth=1, linestyle='--', alpha=0.5)
    ax_res.hlines(-1e-6, xmin=lam_win[0], xmax=lam_win[-1], color="k", linewidth=1, linestyle='--', alpha=0.5)
    
    ax_res.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax_res.set_ylabel("residual")
    ax_res.set_yscale("symlog", linthresh=1e-6)
    
    major_ticks = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]
    ax_res.set_yticks(major_ticks)
    ax_res.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
    ax_res.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

fig.tight_layout()
fig.subplots_adjust(hspace=0.05, wspace=0.2)

plt.show()

# Check against analytical results varying scale factors

### Define wavelength range and lines to study

In [ ]:
wavelengths = jnp.linspace(1200, 1220, 2**18)  # in Angstroms
line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "SiIII-1206"])

### Define fields

In [ ]:
L_box = 0.01

T0 = jnp.finfo(jnp.dtype('float32')).tiny
v0 = 0.0
n0 = 1e12

fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=T0, T0_max=T0, L_box=L_box)),
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=v0, v0_max=v0, L_box=L_box)),
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0)),
        "number_density": ScalarField(toy_samplers.number_density_sampler(n0_min=n0, n0_max=n0, L_box=L_box)),
    },
    "SiIII": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=T0, T0_max=T0, L_box=L_box)),
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=v0, v0_max=v0, L_box=L_box)),
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0)),
        "number_density": ScalarField(toy_samplers.number_density_sampler(n0_min=n0, n0_max=n0, L_box=L_box)),
    }
})

### Define cosmology

In [ ]:
# Create cosmology object
cosmology = Cosmology(
    H0=phys.H0,
    Omega_r_0=1.0,  # Radiation density (CMB + neutrinos)
    Omega_m_0=0.0,
    Omega_k_0=0.0,
    Omega_lambda_0=0.0,
)

### Define skewers

In [ ]:
# create a skewer that go from x=0 to x=L_box with y=L_box/2 and z=L_box/2

N_sk = 2
N_ppsk = 84

_points = jnp.array([
    jnp.linspace(0.0, L_box, N_ppsk),
    L_box/2 * jnp.ones(N_ppsk),
    L_box/2 * jnp.ones(N_ppsk)
]).T

_points = jnp.tile(_points[None, ...], (N_sk, 1, 1))

In [ ]:
skewer_bundle = SkewerBundle(
    spatial_points=_points,
    a0=[1.0, 9.9766642e-01],                                      # scale factor at which ICs are defined
    x0=[0.0, 0.0],                                                # comoving distance at which ICs are defined, default: 0.0
    ICs_defined_at_index=[-1, 0],                                 # indices along the skewer where ICs are defined, (-1 means last point, 0 means first point), default: -1
    ICs_at_emission_or_absorption=["absorption", "emission"],     # whether the ICs are defined at emission or absorption, default: 'absorption'
    cosmologies=cosmology                                         # when manual_scale_factors are not provided, cosmologies must be defined
)

### Compute optical depth

In [ ]:
optical_depth = od.compute_optical_depth(
    skewer_bundle,
    wavelengths,
    line_bundle,
    fields
)
F_over_F0 = jnp.exp(-optical_depth)

### Validation plot

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)                             # (N_lambda,)
F_over_F0_np = np.asarray(F_over_F0)                             # (N_sp, N_sk, N_lambda)
# F_over_F0_analytic = np.asarray(F_over_F0_analytic)            # (N_sp, N_sk, N_lambda)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(line_bundle.lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    for i_sk in range(N_sk):
        ax.plot(lambdas_np, F_over_F0_np[i_sp, i_sk, :], color=colors[i_sk], linestyle='-', linewidth=2.0)
        # ax.plot(lambdas_np, F_over_F0_analytic[i_sp, i_sk, :], color=colors[i_sk], linestyle=':', linewidth=6.0)

# Axes formatting
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$F(\lambda) / F_0$")
ax.set_title("Line profiles for different transitions", fontsize=20)

fig.tight_layout()
plt.show()



# Zoom half-width around each line center [Angstrom]
delta_lambda_A = 5.0   # adjust to taste
N_lines = line_bundle.n_lines

fig, axes = plt.subplots(
    nrows=2, ncols=N_lines, 
    figsize=(14, 8), 
    sharex='col',
    sharey='row',
    gridspec_kw=dict(height_ratios=[2, 1])
)

if N_lines == 1:
    axes = axes.reshape(2, 1)

for i_sp, (ax_main, ax_res, line) in enumerate(zip(axes[0], axes[1], line_bundle.lines)):
    lambda0 = line.lambda0_A
  
    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]
    ax_main.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)

    for i_sk in range(N_sk):
        ax_main.plot(lam_win, F_over_F0_np[i_sp, i_sk, mask], color=colors[i_sk], linestyle='-', linewidth=2.0)
        # ax_main.plot(lam_win, F_over_F0_analytic[i_sp, i_sk, mask], color=colors[i_sk], linestyle=':', linewidth=6.0, alpha=0.6)

        # residual = F_over_F0_np[i_sp, i_sk, mask] / F_over_F0_analytic[i_sp, i_sk, mask] - 1
        # ax_res.plot(lam_win, residual, color=colors[i_sk], linestyle='-', linewidth=2.0)

    ax_main.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax_main.set_ylabel(r"$F(\lambda) / F_0\, \left[\mathrm{adim}\right]$")
    ax_res.axhline(0.0, color="k", linewidth=1, alpha=0.5)
    ax_res.fill_between(lam_win, -1e-6, 1e-6, color="grey", alpha=0.2)
    ax_res.hlines(1e-6, xmin=lam_win[0], xmax=lam_win[-1], color="k", linewidth=1, linestyle='--', alpha=0.5)
    ax_res.hlines(-1e-6, xmin=lam_win[0], xmax=lam_win[-1], color="k", linewidth=1, linestyle='--', alpha=0.5)
  
    ax_res.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax_res.set_ylabel("residual")
    ax_res.set_yscale("symlog", linthresh=1e-6)
  
    major_ticks = [-1e-2, -1e-5, 0.0, 1e-5, 1e-2]
    ax_res.set_yticks(major_ticks)
    ax_res.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(sjx.utils.plotting_tools.ten_power_formatter))
    ax_res.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())

fig.tight_layout()
fig.subplots_adjust(hspace=0.05, wspace=0.2)
plt.show()

In [ ]:
def compute_tau_analytical_radiation_domination(
    lambda_vals,                      # (N_lambda,)
    lambda_rest,                      # (N_lines,)
    gamma,                            # (N_lines,)  [1/time]
    f_osc,                            # (N_lines,)  dimensionless oscillator strength
    a0,                               # (N_sk,)     scale factor at observer
    x0,                               # (N_sk,)     comoving dist to observer [length]
    x_em,                             # (N_sk,)     comoving dist to emission [length]
    n0,                               # (N_sk,)     number density at observer [length^-3]
) -> jnp.ndarray:

    # --- broadcast shapes ----------------------------------------------------
    lambda_vals = jnp.asarray(lambda_vals)[None, None, :]     # (1, 1, N_lambda)
    lambda_rest = jnp.asarray(lambda_rest)[:, None, None]     # (N_lines, 1, 1)
    gamma       = jnp.asarray(gamma)[:, None, None]           # (N_lines, 1, 1)
    f_osc       = jnp.asarray(f_osc)[:, None, None]           # (N_lines, 1, 1)
    a0          = jnp.asarray(a0)[None, :, None]              # (1, N_sk, 1)
    x0          = jnp.asarray(x0)[None, :, None]              # (1, N_sk, 1)
    x_em        = jnp.asarray(x_em)[None, :, None]            # (1, N_sk, 1)
    n0          = jnp.asarray(n0)[None, :, None]              # (1, N_sk, 1)

    # --- prefactor -----------------------------------------------------------
    prefactor = (n0 / (jnp.pi * a0 * phys.H0)) * (phys.qe**2 / (4 * phys.epsilon_0 * phys.me * phys.c))*1e3 * f_osc * lambda_vals * 0.1

    # --- helper: arctan term -------------------------------------------------
    def arctan_term(_lambda):
        return jnp.arctan((phys.c / gamma) * (1 / lambda_rest - 1 / _lambda)) * 10.0

    # --- optical depth -------------------------------------------------------------
    return prefactor * (arctan_term(lambda_vals) - arctan_term(lambda_vals * 0.1 * ((phys.H0 / (phys.c * a0)) * (x_em - x0) + 1.0)))

In [ ]:
optical_depth_analytical = compute_tau_analytical_radiation_domination(
    lambda_vals = wavelengths,                       # (N_lambda,)
    lambda_rest = line_bundle.lambda0_A,             # (N_lines,)
    gamma = line_bundle.A21_GHz/(4*jnp.pi),          # (N_lines,)
    f_osc = line_bundle.f_osc,                       # (N_lines,)
    a0 = jnp.array([1.0]),                           # (N_sk,)
    x0 = jnp.array([0.0]),                           # (N_sk,)
    x_em = jnp.array([L_box]),                      # (N_sk,)
    n0 = jnp.array([1e2]),                           # (N_sk,)
)
F_over_F0_analytic = jnp.exp(-optical_depth_analytical)

In [ ]:
optical_depth_analytical

In [ ]:
F_over_F0_analytic

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)                             # (N_lambda,)
F_over_F0_analytic = np.asarray(F_over_F0_analytic[:, 0])              # (N_sp, N_lambda)

colors = np.array(plt.cm.get_cmap("tab10").colors[:F_over_F0_np.shape[0]])

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(line_bundle.lines):
    ax.plot(lambdas_np, F_over_F0_analytic[i_sp, :], color=colors[i_sp], linestyle=':', linewidth=1.0)

fig.tight_layout()
plt.show()

# Check variations with temperature and LoS velocity

In [ ]:
L_box = 1.0

fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=1e0, T0_max=1e6, L_box=L_box)),
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=-0., v0_max=0., L_box=L_box)),
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0)),
        "number_density": ScalarField(toy_samplers.number_density_sampler(n0_min=1e0, n0_max=1e6, L_box=L_box)),
    },
    "SiIII": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_min=1e0, T0_max=1e6, L_box=L_box)),
        "velocity": VectorField(toy_samplers.velocity_sampler(v0_min=-0., v0_max=0., L_box=L_box)),
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0)),
        "number_density": ScalarField(toy_samplers.number_density_sampler(n0_min=1e0, n0_max=1e6, L_box=L_box)),
    }
})

In [ ]:
N_sk = 7
N_ppsk = 32

_points        = aux.make_plane_points(L_box, 'y', L_box/2, N_sk, N_ppsk)
skewer_bundle = SkewerBundle(
    spatial_points=_points,
    a0=1.0,                                                      # scale factor at which ICs are defined, default: 1.0
    x0=0.0,                                                      # comoving distance at which ICs are defined, default: 0.0
    ICs_defined_at_index=-1,                                     # indices along the skewer where ICs are defined, (-1 means last point, 0 means first point), default: -1
    ICs_at_emission_or_absorption="absorption",                  # whether the ICs are defined at emission or absorption, default: 'absorption'
    los_directions=None,                                         # Let SkewerBundle compute los_directions automatically
    manual_scale_factors=jnp.ones_like(_points[None, ..., 0]),   # if provided, must have shape (N_cosmologies, N_skewers, N_points_per_skewer)
    cosmologies=None                                             # when the manual scale factors are not provided, cosmologies must be defined
)
optical_depth = od.compute_optical_depth(
    skewer_bundle,
    wavelengths,
    line_bundle,
    fields,
    compute_mode="vmap",
    num_gpus=None
)

In [ ]:
F_over_F0 = jnp.exp(-optical_depth)

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)           # (N_lambda,)
F_over_F0_np = np.asarray(F_over_F0)                           # (N_sp, N_sk, N_lambda)

colors = np.array(plt.cm.get_cmap("tab10").colors[:N_sk])
linestyles = ['-', '--', ':']

fig, ax = plt.subplots(figsize=(14, 6))

# Plot all profiles on the same axes
for i_sp, line in enumerate(line_bundle.lines):
    ax.axvline(line.lambda0_A, color="grey", linestyle="-.", linewidth=1.5, alpha=1.0)
    ls = linestyles[i_sp % len(linestyles)]
    for i_sk in range(N_sk):
        ax.plot(lambdas_np, F_over_F0_np[i_sp, i_sk, :], color=colors[i_sk], linestyle=ls, linewidth=2.0)

# Axes formatting
ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
ax.set_ylabel(r"$F(\lambda) / F_0\, \left[\mathrm{adim}\right]$")
ax.grid(alpha=0.2, which="both")
ax.set_title("Line profiles for different transitions and temperatures", fontsize=20)

# temp_handles = []
# for i_sk in range(N_sk):
#     _sampled_T = fields.temperature(quadri_vectors[i_sk])
#     lbl = rf"$T = {_sampled_T[0, 0]:.2g}\,\mathrm{{K}}$"
#     temp_handles.append(Line2D([0], [0], color=colors[i_sk], lw=4, linestyle="-", label=lbl))
# leg_T = ax.legend(handles=temp_handles, loc="upper left", fontsize=16, ncols=1)
# ax.add_artist(leg_T)


# Legend for lines (linestyles)
line_handles = []
for i_sp, line in enumerate(line_bundle.lines):
    ls = linestyles[i_sp % len(linestyles)]
    lbl = rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$"
    line_handles.append(Line2D([0], [0], color="k", lw=3, linestyle=ls, label=lbl))
leg_L = ax.legend(handles=line_handles, loc="lower right", fontsize=16, ncol=line_bundle.n_lines)

fig.tight_layout()
plt.show()



# Zoom half-width around each line center [Angstrom]
delta_lambda_A = 0.01   # adjust to taste
fig, axes = plt.subplots(nrows=1, ncols=line_bundle.n_lines, figsize=(14, 6), sharex=False, sharey=True)
if line_bundle.n_lines == 1:
    axes = [axes]  # in case N_sp=1, make it iterable

for i_sp, (ax, line) in enumerate(zip(axes, line_bundle.lines)):
    lambda0 = line.lambda0_A
    ax.axvline(lambda0, color="grey", linestyle="-.", linewidth=1.5)
    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 + delta_lambda_A)
    lam_win = lambdas_np[mask]  # (N_lambda_window,)

    # Plot each temperature (each skewer)
    for i_sk in range(N_sk):
        ax.plot(lam_win, F_over_F0_np[i_sp, i_sk, mask], color=colors[i_sk], linestyle=linestyles[i_sp], linewidth=2.0)

    # Panel title and scaling
    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")
    ax.grid(alpha=0.2, which="both")

axes[0].set_ylabel(r"$F(\lambda) / F_0\, \left[\mathrm{adim}\right]$")
fig.tight_layout()
plt.show()

# tests to explore vmap, pmap and Pytrees

In [ ]:
from jax import vmap, jit, pmap
import jax.numpy as jnp
import matplotlib.pyplot as plt



def make_chi2_computer(model_fn):
    """Factory: model_fn is baked into the returned function."""
    def chi2(data, params):
        model = model_fn(data["x"], params)
        return jnp.sum((data["y"] - model)**2 / data["noise"]**2)
    return chi2

def sinusoid_model(x, params):
    return params["amp"] * jnp.sin(x + params["phase"])


chi2_sinusoid = make_chi2_computer(sinusoid_model)



# Each sightline is a dict of arrays shape (n_points,)
datasets = {
    "x": jnp.array([
        [0.0, 0.5, 1.0, 1.5, 2.0],  # Sightline 0
        [0.0, 0.7, 1.4, 2.1, 2.8],  # Sightline 1
        [0.0, 1.0, 2.0, 3.0, 4.0],  # Sightline 2
    ]),
    "y": jnp.array([
        [0.1, 0.9, 0.2, -0.7, -0.3],  # Observed flux for sightline 0
        [0.0, 0.5, 0.8, 0.3, -0.5],   # Sightline 1
        [0.1, 1.2, 0.9, 0.4, 0.1],    # Sightline 2
    ]),
    "noise": jnp.full((3, 5), 0.05)  # Constant noise for all
}

params_sine = {"amp": 1.0, "phase": 0.0}




vcompute_sinusoid = vmap(chi2_sinusoid, in_axes=(0, None))
chi2_vmap = vcompute_sinusoid(datasets, params_sine)
print(chi2_vmap)

In [ ]:
import jax

In [ ]:
# Simulate 8 GPUs (use real number if available)
n_devices = min(8, jax.local_device_count())
print(f"Available devices: {jax.local_device_count()}, Using: {n_devices}")

# Reshape data: (n_devices, n_sightlines_per_device, ...)
# Pad to multiple of n_devices if needed
n_total = datasets["x"].shape[0]
if n_total % n_devices != 0:
    pad_size = n_devices - (n_total % n_devices)
    datasets = {k: jnp.pad(v, ((0, pad_size), (0, 0))) for k, v in datasets.items()}

datasets_sharded = {
    k: jnp.reshape(v, (n_devices, -1, v.shape[-1])) 
    for k, v in datasets.items()
}
print(f"Sharded shape: {datasets_sharded['x'].shape}")  # (8, 1, 5) if 8 devices

# Create pmapped pipeline
# pmap the vmapped function: one per device
pcompute_sinusoid = pmap(
    vcompute_sinusoid,  # Re-use the vmapped version
    in_axes=(0, None),  # Shard the dataset axis 0, broadcast params
    axis_name="devices"
)

# Run on all devices
chi2_pmap = pcompute_sinusoid(datasets_sharded, params_sine)
print(f"PMAP χ² (sharded): {chi2_pmap}")  # shape: (8, 1)
print(f"PMAP χ² (flattened): {chi2_pmap.flatten()[:n_total]}")